# Camada Silver — Tratamento e Integração

Este notebook implementa o processamento da camada Silver da pipeline de dados de alfabetização.

Os dados brutos são lidos da camada Bronze no Amazon S3 e passam por validações, padronizações e tratamentos de qualidade antes de serem persistidos na Silver.

As transformações são definidas com base nos problemas identificados durante a exploração dos datasets, evitando alterações ou imputações sem justificativa nos dados de origem.

## 1. Leitura dos dados da Bronze

Nesta etapa são definidos os caminhos das camadas Bronze e Silver e realizada a leitura dos seis datasets utilizando Apache Spark.

As quantidades de registros são verificadas para garantir que os dados recebidos pela Silver correspondem às fontes validadas na Bronze.

In [0]:
# Define os caminhos das camadas Bronze e Silver no Amazon S3.

from pyspark.sql import functions as F

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

BRONZE_PATH = f"s3://{BUCKET_NAME}/bronze/"
SILVER_PATH = f"s3://{BUCKET_NAME}/silver/"

print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")

In [0]:
# Mapeia os datasets da Bronze que serão processados na camada Silver.

DATASETS = {
    "avaliacao_alfabetizacao_municipio": f"{BRONZE_PATH}avaliacao_alfabetizacao_municipio.csv",
    "avaliacao_alfabetizacao_uf": f"{BRONZE_PATH}avaliacao_alfabetizacao_uf.csv",
    "avaliacao_alunos": f"{BRONZE_PATH}avaliacao_alunos.csv",
    "meta_alfabetizacao_brasil": f"{BRONZE_PATH}meta_alfabetizacao_brasil.csv",
    "meta_alfabetizacao_municipio": f"{BRONZE_PATH}meta_alfabetizacao_municipio.csv",
    "meta_alfabetizacao_uf": f"{BRONZE_PATH}meta_alfabetizacao_uf.csv"
}

In [0]:
# Carrega os arquivos CSV da Bronze com Spark para iniciar o processamento da Silver.

dataframes_bronze = {}

for nome, caminho in DATASETS.items():
    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(caminho)
        )

        dataframes_bronze[nome] = df
        print(f"[OK] {nome}: {df.count()} registros")

    except Exception as erro:
        print(f"[ERRO] {nome}: {erro}")

## 2. Validação de chaves e duplicidades

As chaves candidatas identificadas durante a exploração são utilizadas para verificar a unicidade dos registros.

In [0]:
# Define as chaves candidatas identificadas durante a exploração para validar duplicidades e unicidade antes dos tratamentos da camada Silver.

CHAVES = {
    "avaliacao_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "avaliacao_alfabetizacao_uf": ["ano", "sigla_uf", "rede"],
    "avaliacao_alunos": ["ano", "id_aluno"],
    "meta_alfabetizacao_brasil": ["ano", "rede"],
    "meta_alfabetizacao_municipio": ["ano", "id_municipio", "rede"],
    "meta_alfabetizacao_uf": ["ano", "sigla_uf", "rede"]
}

In [0]:
# Verifica a existência de registros duplicados considerando as chaves candidatas de cada dataset, sem realizar nenhuma remoção nesta etapa.

resultado_duplicidades = []

for nome, chaves in CHAVES.items():
    df = dataframes_bronze[nome]

    duplicados = (
        df.groupBy(*chaves)
          .count()
          .filter("count > 1")
    )

    qtd_chaves_duplicadas = duplicados.count()

    resultado_duplicidades.append({
        "dataset": nome,
        "chave": " + ".join(chaves),
        "chaves_duplicadas": qtd_chaves_duplicadas
    })

df_duplicidades = spark.createDataFrame(resultado_duplicidades)

display(df_duplicidades.orderBy("dataset"))

## 3. Análise e tratamento de valores nulos

Os valores nulos são analisados antes das transformações para distinguir problemas de qualidade de ausências legítimas da fonte.

In [0]:
# Calcula a quantidade de valores nulos por coluna em cada dataset da Bronze para identificar quais campos precisam de regras de tratamento na Silver.

from pyspark.sql import functions as F
resultado_nulos = []

for nome, df in dataframes_bronze.items():
    total_registros = df.count()

    contagem_nulos = df.select([
        F.sum(F.col(coluna).isNull().cast("int")).alias(coluna)
        for coluna in df.columns
    ]).collect()[0].asDict()

    for coluna, qtd_nulos in contagem_nulos.items():
        if qtd_nulos > 0:
            resultado_nulos.append({
                "dataset": nome,
                "coluna": coluna,
                "nulos": qtd_nulos,
                "percentual_nulos": round(
                    (qtd_nulos / total_registros) * 100, 2
                )
            })

df_nulos = spark.createDataFrame(resultado_nulos)

display(
    df_nulos.orderBy(
        "dataset",
        F.desc("percentual_nulos")
    )
)

In [0]:
# Analisa a relação entre a presença do aluno e os valores nulos de proficiencia e peso_aluno para definir a regra de tratamento na Silver.

df_alunos = dataframes_bronze["avaliacao_alunos"]

(
    df_alunos
    .groupBy("presenca")
    .agg(
        F.count("*").alias("registros"),
        F.sum(F.col("proficiencia").isNull().cast("int"))
            .alias("proficiencia_nula"),
        F.sum(F.col("peso_aluno").isNull().cast("int"))
            .alias("peso_aluno_nulo")
    )
    .orderBy("presenca")
    .display()
)

In [0]:
# Investiga registros de alunos presentes que possuem proficiencia ou peso_aluno nulos para identificar possíveis inconsistências antes de definir o tratamento na Silver.

df_alunos_presentes_nulos = (
    df_alunos
    .filter(
        (F.col("presenca") == 1) &
        (
            F.col("proficiencia").isNull() |
            F.col("peso_aluno").isNull()
        )
    )
)

print(f"Registros encontrados: {df_alunos_presentes_nulos.count()}")
display(df_alunos_presentes_nulos)

In [0]:
# Valida se os valores nulos de proficiencia e peso_aluno estão associados à ausência de preenchimento do caderno de avaliação.

(
    df_alunos
    .groupBy("preenchimento_caderno")
    .agg(
        F.count("*").alias("registros"),
        F.sum(F.col("proficiencia").isNull().cast("int"))
            .alias("proficiencia_nula"),
        F.sum(F.col("peso_aluno").isNull().cast("int"))
            .alias("peso_aluno_nulo")
    )
    .orderBy("preenchimento_caderno")
    .display()
)

In [0]:
# Analisa os valores nulos das proporções de níveis de alfabetização por ano para identificar se a ausência desses dados está associada a períodos específicos.

for nome in [
    "avaliacao_alfabetizacao_municipio",
    "avaliacao_alfabetizacao_uf"
]:
    df = dataframes_bronze[nome]

    colunas_proporcao = [
        coluna for coluna in df.columns
        if coluna.startswith("proporcao_aluno_nivel_")
    ]

    print(f"\nDataset: {nome}")

    (
        df
        .groupBy("ano")
        .agg(
            F.count("*").alias("registros"),
            *[
                F.sum(F.col(coluna).isNull().cast("int"))
                .alias(f"{coluna}_nulos")
                for coluna in colunas_proporcao
            ]
        )
        .orderBy("ano")
        .display()
    )

In [0]:
# Analisa a distribuição dos valores nulos das bases de metas por ano e rede para identificar padrões antes de definir as regras de tratamento da Silver.

for nome in [
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]:
    df = dataframes_bronze[nome]

    colunas_com_nulos = [
        coluna
        for coluna in df.columns
        if df.filter(F.col(coluna).isNull()).limit(1).count() > 0
    ]

    print(f"\nDataset: {nome}")
    print(f"Colunas analisadas: {colunas_com_nulos}")

    (
        df
        .groupBy("ano", "rede")
        .agg(
            F.count("*").alias("registros"),
            *[
                F.sum(F.col(coluna).isNull().cast("int"))
                .alias(f"{coluna}_nulos")
                for coluna in colunas_com_nulos
            ]
        )
        .orderBy("ano", "rede")
        .display()
    )

In [0]:
# Identifica os registros das bases de metas que possuem pelo menos um valor nulo, permitindo analisar quais municípios e UFs concentram essas ausências.

df_meta_municipio = dataframes_bronze["meta_alfabetizacao_municipio"]
df_meta_uf = dataframes_bronze["meta_alfabetizacao_uf"]

colunas_nulos_municipio = [
    "taxa_alfabetizacao",
    "meta_alfabetizacao_2024",
    "nivel_alfabetizacao",
    "percentual_participacao"
]

condicao_municipio = F.lit(False)

for coluna in colunas_nulos_municipio:
    condicao_municipio = condicao_municipio | F.col(coluna).isNull()

display(
    df_meta_municipio
    .filter(condicao_municipio)
    .select(
        "ano",
        "id_municipio",
        "rede",
        *colunas_nulos_municipio
    )
    .orderBy("id_municipio", "ano")
)

In [0]:
# Identifica os registros estaduais que possuem pelo menos um valor nulo nas taxas, participação ou metas de alfabetização.

colunas_nulos_uf = [
    "taxa_alfabetizacao",
    "percentual_participacao",
    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2025",
    "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027",
    "meta_alfabetizacao_2028",
    "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030"
]

condicao_uf = F.lit(False)

for coluna in colunas_nulos_uf:
    condicao_uf = condicao_uf | F.col(coluna).isNull()

display(
    df_meta_uf
    .filter(condicao_uf)
    .select(
        "ano",
        "sigla_uf",
        "rede",
        *colunas_nulos_uf
    )
    .orderBy("sigla_uf", "ano")
)

### Conclusões sobre valores nulos

A análise identificou padrões consistentes nas ausências de dados:

- `avaliacao_alunos`: os valores nulos de `proficiencia` e `peso_aluno` ocorrem exclusivamente quando `preenchimento_caderno = 0`;
- `avaliacao_alfabetizacao_municipio` e `avaliacao_alfabetizacao_uf`: as proporções por nível estão ausentes nos registros de 2023 e disponíveis em 2024;
- `meta_alfabetizacao_municipio`: existem municípios com indicadores e metas não disponíveis na fonte;
- `meta_alfabetizacao_uf`: foram identificadas ausências específicas de indicadores e metas em determinadas UFs;
- `meta_alfabetizacao_brasil`: não apresenta valores nulos.

Como essas ausências possuem contexto identificável e não há base para imputação confiável, os valores nulos serão preservados na camada Silver.

## 4. Padronização e validação dos tipos de dados

Nesta etapa são avaliados os tipos inferidos pelo Spark e os domínios dos principais campos categóricos.

O objetivo é garantir consistência entre os datasets e corrigir apenas tipos ou representações que possam comprometer integrações e análises posteriores.

In [0]:
# Cria os DataFrames de trabalho da camada Silver preservando os dados originais da Bronze para comparações e validações posteriores.

dataframes_silver = {
    nome: df
    for nome, df in dataframes_bronze.items()
}

print(f"[OK] {len(dataframes_silver)} datasets preparados para a camada Silver")

In [0]:
# Exibe os schemas atuais dos datasets para identificar possíveis ajustes de tipos na Silver.

for nome, df in dataframes_bronze.items():
    print("=" * 80)
    print(f"Dataset: {nome}")
    df.printSchema()

In [0]:
# Exibe os valores distintos dos principais campos categóricos para verificar consistência e identificar possíveis necessidades de padronização.

CAMPOS_CATEGORICOS = {
    "avaliacao_alfabetizacao_municipio": ["rede"],
    "avaliacao_alfabetizacao_uf": ["sigla_uf", "rede"],
    "avaliacao_alunos": ["serie", "rede", "presenca", "preenchimento_caderno", "alfabetizado"],
    "meta_alfabetizacao_brasil": ["rede"],
    "meta_alfabetizacao_municipio": ["rede", "nivel_alfabetizacao"],
    "meta_alfabetizacao_uf": ["sigla_uf", "rede"]
}

for nome, colunas in CAMPOS_CATEGORICOS.items():
    df = dataframes_bronze[nome]

    print(f"\nDataset: {nome}")

    for coluna in colunas:
        valores = [
            linha[coluna]
            for linha in df.select(coluna).distinct().orderBy(coluna).collect()
        ]

        print(f"{coluna}: {valores}")

In [0]:
# Analisa a distribuição dos códigos de rede nas bases de avaliação para definir a padronização necessária antes das integrações da Silver.

for nome in [
    "avaliacao_alfabetizacao_municipio",
    "avaliacao_alfabetizacao_uf"
]:
    print(f"\nDataset: {nome}")

    (
        dataframes_bronze[nome]
        .groupBy("rede")
        .agg(
            F.count("*").alias("registros"),
            F.countDistinct("ano").alias("anos")
        )
        .orderBy("rede")
        .display()
    )

In [0]:
# Analisa os códigos de rede presentes na base de alunos para comparar seu domínio com as demais bases de avaliação.

(
    dataframes_bronze["avaliacao_alunos"]
    .groupBy("rede")
    .count()
    .orderBy("rede")
    .display()
)

In [0]:
# Padroniza o tipo das metas de alfabetização para garantir consistência entre os anos e facilitar comparações e integrações na camada Silver.

datasets_metas = [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf"
]

for nome in datasets_metas:
    dataframes_silver[nome] = (
        dataframes_silver[nome]
        .withColumn(
            "meta_alfabetizacao_2030",
            F.col("meta_alfabetizacao_2030").cast("double")
        )
    )

    print(f"[OK] {nome}: meta_alfabetizacao_2030 -> double")

In [0]:
# Valida o tipo da coluna meta_alfabetizacao_2030 após a padronização para confirmar a consistência entre as colunas de metas.

for nome in datasets_metas:
    tipo_coluna = dict(dataframes_bronze[nome].dtypes)["meta_alfabetizacao_2030"]

    print(f"[OK] {nome}: meta_alfabetizacao_2030 = {tipo_coluna}")

In [0]:
# Verifica o comprimento dos identificadores para avaliar se devem ser tratados como códigos textuais na camada Silver.

IDENTIFICADORES = {
    "avaliacao_alfabetizacao_municipio": ["id_municipio"],
    "avaliacao_alunos": ["id_municipio", "id_escola", "id_aluno"],
    "meta_alfabetizacao_municipio": ["id_municipio"]
}

for nome, colunas in IDENTIFICADORES.items():
    df = dataframes_bronze[nome]

    print(f"\nDataset: {nome}")

    for coluna in colunas:
        print(f"Coluna: {coluna}")

        (
            df
            .select(
                F.length(F.col(coluna).cast("string")).alias("tamanho")
            )
            .groupBy("tamanho")
            .count()
            .orderBy("tamanho")
            .display()
        )

In [0]:
# Padroniza os identificadores como strings, pois representam códigos utilizados em relacionamentos e não valores numéricos para cálculo.

CONVERSOES_IDS = {
    "avaliacao_alfabetizacao_municipio": ["id_municipio"],
    "avaliacao_alunos": ["id_municipio", "id_escola", "id_aluno"],
    "meta_alfabetizacao_municipio": ["id_municipio"]
}

for nome, colunas in CONVERSOES_IDS.items():
    for coluna in colunas:
        dataframes_silver[nome] = (
            dataframes_silver[nome]
            .withColumn(coluna, F.col(coluna).cast("string"))
        )

    print(f"[OK] {nome}: identificadores convertidos para string")

In [0]:
# Confirma os tipos dos identificadores após a padronização.

for nome, colunas in CONVERSOES_IDS.items():
    tipos = dict(dataframes_bronze[nome].dtypes)

    print(f"\nDataset: {nome}")

    for coluna in colunas:
        print(f"[OK] {coluna}: {tipos[coluna]}")

In [0]:
# Verifica possíveis espaços extras nos campos textuais antes da padronização.

CAMPOS_TEXTO = {
    "avaliacao_alfabetizacao_uf": ["sigla_uf"],
    "meta_alfabetizacao_brasil": ["rede"],
    "meta_alfabetizacao_municipio": ["rede"],
    "meta_alfabetizacao_uf": ["sigla_uf", "rede"]
}

for nome, colunas in CAMPOS_TEXTO.items():
    df = dataframes_silver[nome]

    print(f"\nDataset: {nome}")

    for coluna in colunas:
        qtd_espacos = (
            df
            .filter(F.col(coluna) != F.trim(F.col(coluna)))
            .count()
        )

        print(f"{coluna}: {qtd_espacos} registros com espaços extras")

In [0]:
# Padroniza campos textuais removendo espaços extras e garantindo siglas de UF em letras maiúsculas.

for nome, colunas in CAMPOS_TEXTO.items():

    for coluna in colunas:

        if coluna == "sigla_uf":
            dataframes_silver[nome] = (
                dataframes_silver[nome]
                .withColumn(
                    coluna,
                    F.upper(F.trim(F.col(coluna)))
                )
            )

        else:
            dataframes_silver[nome] = (
                dataframes_silver[nome]
                .withColumn(
                    coluna,
                    F.trim(F.col(coluna))
                )
            )

print("[OK] Padronização dos campos textuais concluída.")

In [0]:
# Confirma os valores dos campos textuais após a padronização.

for nome, colunas in CAMPOS_TEXTO.items():
    print(f"\nDataset: {nome}")

    for coluna in colunas:
        valores = [
            linha[coluna]
            for linha in (
                dataframes_silver[nome]
                .select(coluna)
                .distinct()
                .orderBy(coluna)
                .collect()
            )
        ]

        print(f"{coluna}: {valores}")

## 5. Validação de qualidade dos dados

Nesta etapa são aplicadas regras de validação sobre os dados já padronizados da Silver.

O objetivo é identificar valores fora dos domínios esperados, faixas percentuais inválidas e inconsistências que possam comprometer análises posteriores. As validações são realizadas antes da integração e persistência dos dados.

In [0]:
# Valida se taxas, percentuais, metas e proporções estão dentro da faixa esperada de 0 a 100.

resultado_faixas = []

for nome, df in dataframes_silver.items():

    colunas_percentuais = [
        coluna
        for coluna in df.columns
        if (
            coluna.startswith("taxa_")
            or coluna.startswith("percentual_")
            or coluna.startswith("meta_alfabetizacao_")
            or coluna.startswith("proporcao_aluno_")
        )
    ]

    for coluna in colunas_percentuais:

        invalidos = (
            df
            .filter(
                F.col(coluna).isNotNull()
                & (
                    (F.col(coluna) < 0)
                    | (F.col(coluna) > 100)
                )
            )
            .count()
        )

        resultado_faixas.append({
            "dataset": nome,
            "coluna": coluna,
            "registros_invalidos": invalidos
        })

df_validacao_faixas = spark.createDataFrame(resultado_faixas)

display(
    df_validacao_faixas
    .filter(F.col("registros_invalidos") > 0)
    .orderBy("dataset", "coluna")
)

In [0]:
# Resume o resultado das validações de faixa numérica realizadas na Silver.

total_validacoes = df_validacao_faixas.count()

total_com_erro = (
    df_validacao_faixas
    .filter(F.col("registros_invalidos") > 0)
    .count()
)

if total_com_erro == 0:
    print(
        f"[OK] {total_validacoes} validações executadas. "
        "Nenhum valor fora da faixa de 0 a 100 foi identificado."
    )
else:
    print(
        f"[ATENÇÃO] {total_com_erro} validações apresentaram "
        "registros fora da faixa esperada."
    )

In [0]:
# Valida se taxas, percentuais, metas e proporções estão dentro da faixa esperada de 0 a 100.

resultado_faixas = []

for nome, df in dataframes_silver.items():

    colunas_percentuais = [
        coluna
        for coluna in df.columns
        if (
            coluna.startswith("taxa_")
            or coluna.startswith("percentual_")
            or coluna.startswith("meta_alfabetizacao_")
            or coluna.startswith("proporcao_aluno_")
        )
    ]

    for coluna in colunas_percentuais:
        invalidos = (
            df
            .filter(
                F.col(coluna).isNotNull()
                & (
                    (F.col(coluna) < 0)
                    | (F.col(coluna) > 100)
                )
            )
            .count()
        )

        resultado_faixas.append({
            "dataset": nome,
            "coluna": coluna,
            "registros_invalidos": invalidos
        })

df_validacao_faixas = spark.createDataFrame(resultado_faixas)

display(
    df_validacao_faixas
    .filter(F.col("registros_invalidos") > 0)
    .orderBy("dataset", "coluna")
)

In [0]:
# Resume o resultado das validações de faixa numérica realizadas na Silver.

total_validacoes = df_validacao_faixas.count()

total_com_erro = (
    df_validacao_faixas
    .filter(F.col("registros_invalidos") > 0)
    .count()
)

if total_com_erro == 0:
    print(
        f"[OK] {total_validacoes} validações executadas. "
        "Nenhum valor fora da faixa de 0 a 100 foi identificado."
    )
else:
    print(
        f"[ATENÇÃO] {total_com_erro} validações apresentaram "
        "registros fora da faixa esperada."
    )

### Resumo das validações de qualidade

Consolida os principais controles de qualidade executados sobre os datasets da camada Silver, facilitando o monitoramento e a geração de evidências da pipeline.

In [0]:
# Consolida as principais métricas de qualidade da Silver em uma única visão para monitoramento e geração de evidências.

resumo_qualidade = []

for nome in dataframes_silver.keys():

    duplicidades = next(
        item["chaves_duplicadas"]
        for item in resultado_duplicidades
        if item["dataset"] == nome
    )

    faixas_invalidas = sum(
        item["registros_invalidos"]
        for item in resultado_faixas
        if item["dataset"] == nome
    )

    status = (
        "OK"
        if duplicidades == 0 and faixas_invalidas == 0
        else "ATENÇÃO"
    )

    resumo_qualidade.append({
        "dataset": nome,
        "chaves_duplicadas": duplicidades,
        "valores_fora_faixa": faixas_invalidas,
        "status": status
    })

df_resumo_qualidade = spark.createDataFrame(resumo_qualidade)

display(df_resumo_qualidade.orderBy("dataset"))

## 6. Integridade entre datasets

Nesta etapa são verificadas as relações entre as bases de avaliação e metas, considerando suas granularidades e chaves de negócio.

O objetivo é identificar registros sem correspondência antes de realizar integrações, evitando joins que possam gerar perdas ou multiplicações indevidas.

In [0]:
# Valida a correspondência entre municípios presentes nas bases de avaliação e de metas, considerando ano e id_municipio.

avaliacao_municipio = dataframes_silver["avaliacao_alfabetizacao_municipio"]
meta_municipio = dataframes_silver["meta_alfabetizacao_municipio"]

chaves_avaliacao_municipio = (
    avaliacao_municipio
    .select("ano", "id_municipio")
    .distinct()
)

chaves_meta_municipio = (
    meta_municipio
    .select("ano", "id_municipio")
    .distinct()
)

sem_meta_municipio = (
    chaves_avaliacao_municipio
    .join(
        chaves_meta_municipio,
        ["ano", "id_municipio"],
        "left_anti"
    )
)

sem_avaliacao_municipio = (
    chaves_meta_municipio
    .join(
        chaves_avaliacao_municipio,
        ["ano", "id_municipio"],
        "left_anti"
    )
)

print(
    f"Avaliação sem meta correspondente: "
    f"{sem_meta_municipio.count()}"
)

print(
    f"Meta sem avaliação correspondente: "
    f"{sem_avaliacao_municipio.count()}"
)

In [0]:
# Valida a correspondência entre UFs presentes nas bases de avaliação e de metas, considerando ano e sigla_uf.

avaliacao_uf = dataframes_silver["avaliacao_alfabetizacao_uf"]
meta_uf = dataframes_silver["meta_alfabetizacao_uf"]

chaves_avaliacao_uf = (
    avaliacao_uf
    .select("ano", "sigla_uf")
    .distinct()
)

chaves_meta_uf = (
    meta_uf
    .select("ano", "sigla_uf")
    .distinct()
)

sem_meta_uf = (
    chaves_avaliacao_uf
    .join(
        chaves_meta_uf,
        ["ano", "sigla_uf"],
        "left_anti"
    )
)

sem_avaliacao_uf = (
    chaves_meta_uf
    .join(
        chaves_avaliacao_uf,
        ["ano", "sigla_uf"],
        "left_anti"
    )
)

print(
    f"Avaliação sem meta correspondente: "
    f"{sem_meta_uf.count()}"
)

print(
    f"Meta sem avaliação correspondente: "
    f"{sem_avaliacao_uf.count()}"
)

In [0]:
# Analisa os registros municipais sem correspondência entre avaliação e metas para identificar se as diferenças de cobertura estão concentradas por ano.

print("Avaliação sem meta, por ano:")
(
    sem_meta_municipio
    .groupBy("ano")
    .count()
    .orderBy("ano")
    .display()
)

print("Meta sem avaliação, por ano:")
(
    sem_avaliacao_municipio
    .groupBy("ano")
    .count()
    .orderBy("ano")
    .display()
)

In [0]:
# Analisa as UFs sem correspondência entre avaliação e metas para identificar diferenças de cobertura antes da integração.

print("Avaliação sem meta:")
display(
    sem_meta_uf
    .orderBy("ano", "sigla_uf")
)

print("Meta sem avaliação:")
display(
    sem_avaliacao_uf
    .orderBy("ano", "sigla_uf")
)

In [0]:
# Verifica quantos registros existem por ano e município nas bases de avaliação e metas antes da integração, evitando multiplicação indevida durante o join.

print("Avaliação municipal:")

(
    avaliacao_municipio
    .groupBy("ano", "id_municipio")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
    .display()
)

print("Meta municipal:")

(
    meta_municipio
    .groupBy("ano", "id_municipio")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
    .display()
)

In [0]:
# Verifica quantos registros existem por ano e UF nas bases de avaliação e metas antes da integração.

print("Avaliação UF:")

(
    avaliacao_uf
    .groupBy("ano", "sigla_uf")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
    .display()
)

print("Meta UF:")

(
    meta_uf
    .groupBy("ano", "sigla_uf")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
    .display()
)

### Decisão de integração

As validações demonstraram diferenças de cobertura e granularidade entre as bases de avaliação e metas.

As bases de avaliação possuem múltiplos registros por ano e localidade devido à desagregação por rede de ensino, enquanto as bases de metas possuem granularidade mais consolidada.

Uma integração direta na Silver poderia replicar valores de metas entre diferentes registros de avaliação e alterar semanticamente a granularidade das fontes.

Por esse motivo, os datasets serão mantidos separados na camada Silver, preservando suas granularidades originais. As integrações necessárias para comparação entre resultados observados e metas serão realizadas posteriormente na camada Gold, de acordo com os indicadores analíticos definidos.

## 7. Persistência da Silver

Nesta etapa, os datasets tratados e padronizados são persistidos na camada Silver do Amazon S3.

Os dados serão armazenados em formato Parquet, permitindo armazenamento colunar, compressão e leitura eficiente com Apache Spark.

A camada Bronze permanece preservada em seu formato CSV original.

In [0]:
# Valida o caminho de destino antes da persistência para evitar sobrescrita acidental dos dados brutos da camada Bronze.

assert SILVER_PATH != BRONZE_PATH, \
    "ERRO: Silver e Bronze apontam para o mesmo caminho."

assert "/silver/" in SILVER_PATH, \
    "ERRO: destino não corresponde à camada Silver."

print("[OK] Caminho de escrita da Silver validado.")

In [0]:
for nome, df in dataframes_silver.items():
    caminho_destino = f"{SILVER_PATH}{nome}/"

    (
        df.write
        .mode("overwrite")
        .parquet(caminho_destino)
    )

    print(f"[OK] {nome} gravado em {caminho_destino}")

## 8. Validação Bronze × Silver

Nesta etapa são comparadas as métricas da Bronze e da Silver para garantir que as transformações aplicadas não provocaram perda inesperada de dados.

São validados registros, colunas, nulos relevantes, chaves e tipos após o processamento.

In [0]:
# Relê os datasets persistidos em Parquet para validar a escrita e a disponibilidade dos dados da camada Silver no Amazon S3.

dataframes_silver_persistidos = {}

for nome in dataframes_silver.keys():
    caminho = f"{SILVER_PATH}{nome}/"

    df = spark.read.parquet(caminho)

    dataframes_silver_persistidos[nome] = df

    print(
        f"[OK] {nome}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

In [0]:
# Compara quantidade de registros e colunas entre Bronze e Silver para identificar possíveis perdas ou alterações estruturais inesperadas.

comparacao_bronze_silver = []

for nome in dataframes_bronze.keys():

    df_bronze = dataframes_bronze[nome]
    df_silver = dataframes_silver_persistidos[nome]

    registros_bronze = df_bronze.count()
    registros_silver = df_silver.count()

    colunas_bronze = len(df_bronze.columns)
    colunas_silver = len(df_silver.columns)

    comparacao_bronze_silver.append({
        "dataset": nome,
        "registros_bronze": registros_bronze,
        "registros_silver": registros_silver,
        "diferenca_registros": registros_silver - registros_bronze,
        "colunas_bronze": colunas_bronze,
        "colunas_silver": colunas_silver
    })

df_comparacao_bronze_silver = spark.createDataFrame(
    comparacao_bronze_silver
)

display(
    df_comparacao_bronze_silver.orderBy("dataset")
)

In [0]:
# Valida os tipos alterados durante o processamento da Silver diretamente nos datasets persistidos em Parquet.

TIPOS_ESPERADOS = {
    "avaliacao_alfabetizacao_municipio": {
        "id_municipio": "string"
    },
    "avaliacao_alunos": {
        "id_municipio": "string",
        "id_escola": "string",
        "id_aluno": "string"
    },
    "meta_alfabetizacao_brasil": {
        "meta_alfabetizacao_2030": "double"
    },
    "meta_alfabetizacao_municipio": {
        "id_municipio": "string",
        "meta_alfabetizacao_2030": "double"
    },
    "meta_alfabetizacao_uf": {
        "meta_alfabetizacao_2030": "double"
    }
}

erros_tipo = []

for nome, colunas in TIPOS_ESPERADOS.items():
    tipos_atuais = dict(dataframes_silver_persistidos[nome].dtypes)

    for coluna, tipo_esperado in colunas.items():
        tipo_atual = tipos_atuais[coluna]

        if tipo_atual == tipo_esperado:
            print(f"[OK] {nome}.{coluna}: {tipo_atual}")
        else:
            erros_tipo.append(
                f"{nome}.{coluna}: esperado {tipo_esperado}, encontrado {tipo_atual}"
            )
            print(
                f"[ERRO] {nome}.{coluna}: "
                f"esperado {tipo_esperado}, encontrado {tipo_atual}"
            )

if not erros_tipo:
    print("\n[OK] Todos os tipos padronizados foram persistidos corretamente.")

In [0]:
# Valida novamente a unicidade das chaves nos datasets persistidos para garantir que o processamento da Silver não criou duplicidades.

duplicidades_silver = []

for nome, chaves in CHAVES.items():
    df = dataframes_silver_persistidos[nome]

    qtd_duplicadas = (
        df
        .groupBy(*chaves)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicidades_silver.append({
        "dataset": nome,
        "chaves_duplicadas": qtd_duplicadas
    })

df_duplicidades_silver = spark.createDataFrame(duplicidades_silver)

display(df_duplicidades_silver.orderBy("dataset"))

## 9. Metadados e execução da Silver

Nesta etapa são consolidadas informações sobre o processamento dos datasets, incluindo quantidade de registros de entrada e saída, status e horário da execução.

Esses metadados serão utilizados como evidência da execução da pipeline e como base para monitoramento.

In [0]:
# Consolida os metadados da execução da Silver, registrando quantidade de entrada, saída, formato, status e horário do processamento.

from datetime import datetime, timezone

data_hora_processamento = datetime.now(timezone.utc)

metadados_silver = []

for nome in dataframes_bronze.keys():

    registros_entrada = dataframes_bronze[nome].count()
    registros_saida = dataframes_silver_persistidos[nome].count()

    metadados_silver.append({
        "dataset": nome,
        "registros_entrada": registros_entrada,
        "registros_saida": registros_saida,
        "diferenca_registros": registros_saida - registros_entrada,
        "camada": "silver",
        "formato": "parquet",
        "status": "SUCESSO",
        "data_hora_processamento_utc": data_hora_processamento
    })

df_metadados_silver = spark.createDataFrame(metadados_silver)

display(df_metadados_silver.orderBy("dataset"))

In [0]:
# Valida se todos os datasets foram processados e persistidos corretamente antes de considerar a execução da Silver concluída.

datasets_esperados = set(DATASETS.keys())
datasets_persistidos = set(dataframes_silver_persistidos.keys())

faltantes = datasets_esperados - datasets_persistidos

if faltantes:
    raise Exception(
        f"Falha na Silver. Datasets não persistidos: {faltantes}"
    )

registros_com_diferenca = (
    df_metadados_silver
    .filter(F.col("diferenca_registros") != 0)
    .count()
)

if registros_com_diferenca > 0:
    raise Exception(
        "Falha na Silver. Foram identificadas diferenças inesperadas "
        "na quantidade de registros."
    )

print(
    f"[OK] Silver validada: "
    f"{len(datasets_persistidos)}/{len(datasets_esperados)} datasets "
    "processados e persistidos sem perda de registros."
)